#### 1.1 Import Data and Required Packages
Importing Pandas, Numpy, Matplotlib, Seaborn, and Warnings Library.

In [ ]:
#basic library
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

# all regression model 
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')


#### 1.2 Import the CSV Data as Pandas DataFrame

In [3]:
import pandas as pd

# loading csv data 
df1 = pd.read_csv('data/transactions_in_usd.csv')

#cleaning weekday to make it exactly seven days of week 
df1['Weekday'] = df1['Weekday'].str.strip().str.title()

#splitting date to differnt column year , month , day 
df1['Transaction Date'] = pd.to_datetime(df1['Transaction Date'], format='mixed', dayfirst=True)
df1['Year'] = df1['Transaction Date'].dt.year
df1['Month'] = df1['Transaction Date'].dt.month
df1['Day'] = df1['Transaction Date'].dt.day

#feature engineering and adding extra column adding total and average 
df1['Avg Amt Per Withdrawal'] = df1['Total amount Withdrawn'] / df1['No Of Withdrawals']
df1['XYZ Card Share %'] = (df1['No Of XYZ Card Withdrawals'] / df1['No Of Withdrawals']) * 100


df1 = df1.drop(columns=['Transaction Date', 'Avg Amt Per Withdrawal'])



#### 1.3 Preparing X and y Variables

In [4]:
#splitting x and y where x is independent and y is dependent 
X = df1.drop(columns=['Total amount Withdrawn'])
y = df1['Total amount Withdrawn']

# verifying splitting 
X.head()


,ATM Name,No Of Withdrawals,No Of XYZ Card Withdrawals,No Of Other Card Withdrawals,Amount withdrawn XYZ Card,Amount withdrawn Other Card,Weekday,Festival Religion,Working Day,Holiday Sequence,Year,Month,Day,XYZ Card Share %
0,Big Street ATM,50,20,30,892.38,1756.94,Saturday,H,H,WHH,2011,1,1,40.000000
1,Mount Road ATM,253,67,186,5797.26,10635.80,Saturday,C,H,WHH,2011,1,1,26.482213
2,Airport ATM,98,56,42,7440.78,3331.98,Saturday,C,H,WHH,2011,1,1,57.142857
3,KK Nagar ATM,265,159,106,11397.64,8831.78,Saturday,C,H,WHH,2011,1,1,60.000000
4,Christ College ATM,74,25,49,3171.48,2985.30,Saturday,C,H,WHH,2011,1,1,33.783784


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

#separating numerical and categorical column to apply some transformation for different dtypes 
categorical_cols = ['ATM Name', 'Weekday', 'Festival Religion', 'Working Day', 'Holiday Sequence']
numerical_cols = ['Year', 'Month', 'Day', 'XYZ Card Share %'] # Add any other numeric columns here

#building transformation process
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ])

#splitting data 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# transforming the column using transformation pipeline 
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Training shape: {X_train_processed.shape}")
print(f"Testing shape: {X_test_processed.shape}")


Training shape: (9271, 26)
Testing shape: (2318, 26)


In [7]:
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

#list of all regression model 
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(alpha=1.0),
    "Ridge": Ridge(alpha=1.0),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest Regressor": RandomForestRegressor(random_state=42),
    "AdaBoost Regressor": AdaBoostRegressor(random_state=42),
    "Support Vector Regressor": SVR(),
    "XGBRegressor": XGBRegressor(random_state=42),
    "CatBoost Regressor": CatBoostRegressor(verbose=0, random_state=42)
}

def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    rmse = root_mean_squared_error(true, predicted) # root_mean_squared_error does not take 'squared' argument
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

model_list = []
r2_list = []

print(f"{'Model Name':<30} | {'RMSE':<12} | {'MAE':<12} | {'R2 Score':<10}")
print("-" * 75)

for name, model in models.items():

    model.fit(X_train_processed, y_train)
 
    y_pred = model.predict(X_test_processed)
    

    mae, rmse, r2_square = evaluate_model(y_test, y_pred)
    
    print(f"{name:<30} | {rmse:<12.4f} | {mae:<12.4f} | {r2_square:<10.4f}")
    
    model_list.append(name)
    r2_list.append(r2_square)
    
results_df = pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"], ascending=False).reset_index(drop=True)


Model Name                     | RMSE         | MAE          | R2 Score  
---------------------------------------------------------------------------
Linear Regression              | 4412.5750    | 3328.7091    | 0.3626    
Lasso                          | 4412.6246    | 3327.3661    | 0.3626    
Ridge                          | 4412.3463    | 3328.2356    | 0.3627    
K-Neighbors Regressor          | 3746.0799    | 2696.0504    | 0.5406    
Decision Tree                  | 4301.5682    | 3101.0161    | 0.3943    
Random Forest Regressor        | 3155.0235    | 2261.0727    | 0.6741    
AdaBoost Regressor             | 4262.2638    | 3353.1814    | 0.4053    
Support Vector Regressor       | 5521.6231    | 4041.7170    | 0.0019    
XGBRegressor                   | 3153.6996    | 2288.3256    | 0.6744    
CatBoost Regressor             | 2992.0336    | 2183.3806    | 0.7069    


#### Hyperparamteter tuning 

In [8]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor

cat_model = CatBoostRegressor(verbose=0, random_state=42, thread_count=-1)


param_distributions = {
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'border_count': [32, 64, 128, 255]
}


random_search = RandomizedSearchCV(
    estimator=cat_model,
    param_distributions=param_distributions,
    n_iter=10,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

print("Tuning CatBoost with parameters")
random_search.fit(X_train_processed, y_train)

print("\n=== Hyperparameter Tuning Complete ===")
print(f"Best R2 Score found during search: {random_search.best_score_:.4f}")
print("Best Parameters:", random_search.best_params_)

best_cat_model = random_search.best_estimator_


Tuning CatBoost with parameters

=== Hyperparameter Tuning Complete ===
Best R2 Score found during search: 0.6802
Best Parameters: {'learning_rate': 0.05, 'border_count': 255}
